# Bab 17: Aturan Asosiasi

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab17/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab17")
print("Folder kerja:", os.getcwd())

## Listing 17.1: Apriori dengan tangan pada sembilan struk

`kode/bab17/apriori_mini.py`

In [ ]:
from itertools import combinations

struk = [{"gula aren", "roti", "pisang"}, {"gula aren", "roti"},
         {"americano", "croissant"}, {"gula aren", "pisang"},
         {"gula aren", "roti", "teh"}, {"americano", "croissant", "roti"},
         {"gula aren", "roti", "pisang"}, {"teh", "pisang"},
         {"gula aren", "teh"}]
n, minsup = len(struk), 0.2
sup = lambda s: sum(s <= t for t in struk) / n

item = sorted(set().union(*struk))
L = [frozenset([i]) for i in item if sup({i}) >= minsup]
semua = {s: sup(s) for s in L}
k = 1
print(f"L1: " + ", ".join(f"{set(s)} {sup(s):.2f}" for s in L))
while L:
    k += 1
    C = {a | b for a in L for b in L if len(a | b) == k}
    C = {c for c in C if all(frozenset(s) in semua
                             for s in combinations(c, k - 1))}
    L = [c for c in C if sup(c) >= minsup]
    if C:
        print(f"C{k}: {len(C)} kandidat; L{k}: " + ", ".join(
            f"{sorted(s)} {sup(s):.2f}" for s in sorted(L, key=sorted)))
    semua.update({s: sup(s) for s in L})

print("\naturan dengan confidence >= 0.6:")
for s in [s for s in semua if len(s) >= 2]:
    for r in range(1, len(s)):
        for a in map(frozenset, combinations(s, r)):
            conf = semua[s] / semua[a]
            lift = conf / semua[s - a]
            if conf >= 0.6:
                print(f"  {sorted(a)} -> {sorted(s - a)}: sup {semua[s]:.2f}"
                      f" conf {conf:.2f} lift {lift:.2f}")

## Listing 17.2: Apriori dan FP-Growth pada seluruh struk

`kode/bab17/fpgrowth.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab17')
import time
from mlxtend.frequent_patterns import apriori, fpgrowth
from siapkan import keranjang, muat_struk

t = muat_struk()
B = keranjang(t)
print(f"{B.shape[0]} struk, {B.shape[1]} menu; rata-rata "
      f"{B.sum(1).mean():.2f} item per struk; kepadatan {B.values.mean():.3f}")
for minsup in [0.01, 0.003, 0.001]:
    for nama, f in [("Apriori", apriori), ("FP-Growth", fpgrowth)]:
        t0 = time.perf_counter()
        hasil = f(B, min_support=minsup, use_colnames=True)
        print(f"minsup {minsup:5.3f} {nama:9s}: {len(hasil):4d} itemset, "
              f"{time.perf_counter() - t0:5.2f} dtk")

## Listing 17.3: Ukuran asosiasi untuk Kopi Susu Gula Aren

`kode/bab17/ukuran.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab17')
import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth
from siapkan import keranjang, muat_struk

pd.set_option("display.width", 120)
t = muat_struk()
for nama, minimal in [("semua struk", 1), ("struk >= 2 item", 2)]:
    B = keranjang(t, minimal=minimal)
    f = fpgrowth(B, min_support=0.002, use_colnames=True)
    r = association_rules(f, metric="lift", min_threshold=0)
    g = r[r["antecedents"] == frozenset({"Kopi Susu Gula Aren"})]
    g = g.assign(konsekuen=g["consequents"].map(lambda s: ", ".join(s)))
    print(f"{nama}: {len(B)} struk; lift semua aturan "
          f"{r['lift'].min():.2f}-{r['lift'].max():.2f}")
    print(g.sort_values("confidence", ascending=False)[
        ["konsekuen", "support", "confidence", "lift", "kulczynski"]]
        .head(5).round(3).to_string(index=False))
    print()

## Listing 17.4: Studi kasus aturan berkonteks untuk paket menu

`kode/bab17/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab17')
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth
from siapkan import keranjang, muat_struk

pd.set_option("display.width", 120)
t = muat_struk()
B = keranjang(t, konteks=True, minimal=2)
f = fpgrowth(B, min_support=0.002, use_colnames=True)
r = association_rules(f, metric="lift", min_threshold=1.1)
r = r[r["antecedents"].map(lambda s: any(i.startswith("[") for i in s))]
r = r.assign(jika=r["antecedents"].map(lambda s: ", ".join(sorted(s))),
             maka=r["consequents"].map(lambda s: ", ".join(sorted(s))))
print(f"{len(B)} struk berisi >= 2 item; {len(r)} aturan berkonteks:")
print(r.sort_values("lift", ascending=False)[
    ["jika", "maka", "support", "confidence", "lift"]].head(12)
    .round(3).to_string(index=False))

pagi = B["[pagi]"]
a, c = B["Americano"], B["Croissant Mentega"]
def lift(m):
    return (a[m] & c[m]).mean() / (a[m].mean() * c[m].mean())
print(f"\nlift Americano & Croissant: semua {lift(pagi | ~pagi):.2f}, "
      f"pagi {lift(pagi):.2f}, bukan pagi {lift(~pagi):.2f}")

g = (B[B["Kopi Susu Gula Aren"]].drop(columns=["[pagi]", "[berbuka]",
                                               "Kopi Susu Gula Aren"])
     .mean().sort_values(ascending=False))
dasar = B[~B["Kopi Susu Gula Aren"]].drop(
    columns=["[pagi]", "[berbuka]", "Kopi Susu Gula Aren"]).mean()
print("\nteman Kopi Susu Gula Aren (struk >= 2 item):")
print(pd.DataFrame({"bersama gula aren": g, "tanpa gula aren": dasar[g.index],
                    "rasio": g / dasar[g.index]}).head(6).round(3)
      .to_string())
semua = association_rules(fpgrowth(B, min_support=0.002, use_colnames=True),
                          metric="lift", min_threshold=0)
semua[["support", "confidence", "lift"]].round(4).to_csv(
    "gambar/bab17/aturan.dat", sep=" ", index=False)